<a href="https://colab.research.google.com/github/osvaldoferrel/baileFerrel/blob/master/Proyecto1_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto 1: Explorador de llaves y puertas

## Búsqueda en espacios de estados

**Alumno:** Osvaldo Ferrel Sánchez

**Problema:** Explorador de un laberinto con llaves y puertas

En este proyecto se implementa un problema de búsqueda en el que un explorador debe recorrer un laberinto hasta llegar a una salida.

Durante el recorrido existen llaves y puertas. Para poder atravesar una puerta, el explorador debe haber recogido previamente la llave correspondiente.

Se comparan diferentes estrategias de búsqueda:

- BFS
- DFS
- A* con h = 0
- A* con distancia Manhattan

El objetivo es observar cómo se comportan estos algoritmos cuando aumenta la dificultad del laberinto.

https://github.com/osvaldoferrel/baileFerrel

# 1. Introducción

Los algoritmos de búsqueda son una parte importante de la Inteligencia Artificial, ya que permiten encontrar soluciones explorando diferentes estados de un problema.

En este proyecto se utiliza un laberinto como ejemplo. El explorador comienza en una posición determinada y debe llegar hasta la salida.

El problema se vuelve más interesante porque existen llaves y puertas. Una puerta no puede atravesarse si el explorador no posee la llave correspondiente.

Por esta razón, no es suficiente conocer únicamente la posición del explorador. También es necesario conocer qué llaves ha recogido.

# 2. Problema elegido y por qué me interesa

El problema elegido consiste en un explorador que debe recorrer un laberinto hasta encontrar la salida.

En el camino existen paredes, llaves y puertas. El explorador puede moverse hacia arriba, abajo, izquierda o derecha.

Me interesó este problema porque parece sencillo al principio, pero la presencia de las llaves hace que una misma posición pueda representar situaciones diferentes.

Por ejemplo, llegar a una posición sin tener una llave puede impedir avanzar por una puerta, mientras que llegar a la misma posición teniendo la llave puede permitir continuar.

Esto hace que el problema sea adecuado para estudiar diferentes métodos de búsqueda.

# 3. Representación del estado

El estado del problema está formado por dos elementos:

    (posición, llaves_recogidas)

La posición indica dónde se encuentra actualmente el explorador.

Las llaves indican cuáles ha recogido durante el recorrido.

Por ejemplo:

    ((4, 6), {'K1', 'K2'})

significa que el explorador está en la fila 4, columna 6 y tiene las llaves K1 y K2.

No se guarda el número de movimientos realizados dentro del estado, porque esto no cambia las posibilidades de movimiento.

De esta manera se mantiene un estado mínimo y solamente se guarda la información necesaria para resolver el problema.

In [ ]:
# Ejemplo de un estado

posicion = (4, 6)
llaves = frozenset(["K1", "K2"])

estado = (posicion, llaves)

print("Estado:")
print(estado)

# 4. Operaciones sucesor

Desde cada posición el explorador puede intentar realizar cuatro movimientos:

- arriba
- abajo
- izquierda
- derecha

Un movimiento solamente es válido cuando:

1. La nueva posición está dentro del laberinto.
2. La nueva posición no es una pared.
3. Si existe una puerta, el explorador tiene la llave necesaria.

Cuando el explorador llega a una casilla que contiene una llave, esta se agrega a las llaves que ya posee.

Cada movimiento tiene un costo de 1.

# 5. Condición de meta

La búsqueda termina cuando el explorador llega a la posición de salida.

No es necesario tener todas las llaves para terminar.

La condición de meta es:

    posición_actual = posición_salida

Esto permite que el explorador termine inmediatamente cuando alcanza la salida.

# 6. Heurística

Para A* se utilizará la distancia Manhattan.

La distancia Manhattan se calcula sumando la diferencia entre las filas y la diferencia entre las columnas.

La fórmula es:

    h(n) = |fila_actual - fila_salida|
           + |columna_actual - columna_salida|

Esta heurística ignora las paredes y las puertas.

Por lo tanto, representa una estimación optimista de la distancia restante, ya que en el problema real pueden ser necesarios movimientos adicionales debido a los obstáculos.

También se utilizará A* con h = 0 para comparar el comportamiento de A* sin información adicional del problema.

In [ ]:
# Ejemplo de distancia Manhattan

posicion_actual = (3, 5)
posicion_salida = (10, 12)

distancia = (
    abs(posicion_actual[0] - posicion_salida[0])
    +
    abs(posicion_actual[1] - posicion_salida[1])
)

print("Distancia Manhattan:", distancia)

In [ ]:
import sys

sys.path.append("src")

from explorador import Laberinto

In [ ]:
mapa_facil = [

    "#########",
    "#E...K.S#",
    "#.#####.#",
    "#.......#",
    "#########"

]

llaves_facil = {
    (1, 5): "K1"
}

puertas_facil = {}

laberinto_facil = Laberinto(
    mapa_facil,
    llaves_facil,
    puertas_facil
)

print("Inicio:", laberinto_facil.inicio)
print("Salida:", laberinto_facil.salida)

In [ ]:
for fila in mapa_facil:
    print(fila)

In [ ]:
solucion, explorados = laberinto_facil.resolver("bfs")

if solucion:

    print("Se encontró una solución.")
    print("Movimientos:", len(solucion.getPath()) - 1)
    print("Estados explorados:", explorados)

else:

    print("No se encontró una solución.")

# 7. Algoritmos utilizados

Se utilizarán cuatro configuraciones de búsqueda.

### BFS

Búsqueda en anchura. Explora primero los estados que están a menor profundidad.

### DFS

Búsqueda en profundidad. Continúa por una rama antes de regresar y probar otras.

### A* con h = 0

Se utiliza A* sin información heurística.

Esto permite observar cómo se comporta la búsqueda cuando solamente se considera el costo acumulado.

### A* con Manhattan

Utiliza el costo acumulado y la distancia Manhattan hasta la salida.

Esta es la heurística propuesta para este problema.

In [ ]:
def comparar_algoritmos(laberinto):

    resultados = []

    pruebas = [
        ("BFS", "bfs", False),
        ("DFS", "dfs", False),
        ("A* h=0", "a*", False),
        ("A* Manhattan", "a*", True)
    ]

    for nombre, estrategia, heuristica in pruebas:

        solucion, explorados = laberinto.resolver(
            estrategia,
            heuristica
        )

        if solucion:

            movimientos = len(solucion.getPath()) - 1

        else:

            movimientos = None

        resultados.append({
            "Algoritmo": nombre,
            "Movimientos": movimientos,
            "Estados explorados": explorados
        })

    return resultados

In [ ]:
resultados_facil = comparar_algoritmos(laberinto_facil)

for resultado in resultados_facil:
    print(resultado)

In [ ]:
import pandas as pd

tabla_facil = pd.DataFrame(resultados_facil)

tabla_facil

# 8. Instancias de prueba

Para observar el comportamiento de los algoritmos se utilizarán diferentes tamaños de problema.

La idea es comenzar con una instancia sencilla y aumentar progresivamente la dificultad.

Se consideran tres niveles:

- Fácil
- Difícil
- Ultra difícil

La instancia ultra difícil utiliza un laberinto de 15 x 15 y hasta cuatro llaves.

El objetivo no es solamente aumentar el tamaño del mapa, sino también aumentar las decisiones que debe tomar el explorador.

In [ ]:
mapa_dificil = [

    "#############",
    "#E....#....S#",
    "#.....#.....#",
    "#..K..D.....#",
    "#...........#",
    "#############"

]

llaves_dificil = {
    (3, 3): "K1"
}

puertas_dificil = {
    (3, 6): "K1"
}

laberinto_dificil = Laberinto(
    mapa_dificil,
    llaves_dificil,
    puertas_dificil
)

for fila in mapa_dificil:
    print(fila)

In [ ]:
resultados_dificil = comparar_algoritmos(
    laberinto_dificil
)

tabla_dificil = pd.DataFrame(
    resultados_dificil
)

tabla_dificil

In [ ]:
import sys

sys.path.append("src")

import ultradificil

In [ ]:
from ultradificil import mapa, llaves, puertas

laberinto_ultra = Laberinto(
    mapa,
    llaves,
    puertas
)

print("Tamaño:")
print(len(mapa), "x", len(mapa[0]))

print("Número de llaves:", len(llaves))
print("Número de puertas:", len(puertas))

In [ ]:
for fila in mapa:
    print("".join(fila))

In [ ]:
resultados_ultra = comparar_algoritmos(
    laberinto_ultra
)

tabla_ultra = pd.DataFrame(
    resultados_ultra
)

tabla_ultra

In [ ]:
print("RESULTADOS DE LA INSTANCIA FÁCIL")
display(tabla_facil)

print("RESULTADOS DE LA INSTANCIA DIFÍCIL")
display(tabla_dificil)

print("RESULTADOS DE LA INSTANCIA ULTRA DIFÍCIL")
display(tabla_ultra)

# 9. Comparación de resultados

Para comparar los algoritmos se utilizaron principalmente dos medidas:

- Número de movimientos de la solución.
- Número de estados explorados.

El número de movimientos permite observar la longitud de la solución encontrada.

El número de estados explorados permite observar cuánto tuvo que trabajar el algoritmo antes de encontrar la solución.

Como los movimientos tienen costo 1, una solución con menos movimientos tiene menor costo.

In [ ]:
import matplotlib.pyplot as plt

tabla = tabla_ultra

plt.figure(figsize=(8, 5))

plt.bar(
    tabla["Algoritmo"],
    tabla["Estados explorados"]
)

plt.title("Estados explorados - Instancia ultra difícil")
plt.xlabel("Algoritmo")
plt.ylabel("Estados explorados")

plt.xticks(rotation=20)

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    tabla["Algoritmo"],
    tabla["Movimientos"]
)

plt.title("Movimientos encontrados - Instancia ultra difícil")
plt.xlabel("Algoritmo")
plt.ylabel("Movimientos")

plt.xticks(rotation=20)

plt.show()

# 10. Factor de ramificación

En el problema se estimó inicialmente un factor de ramificación aproximado de:

    b = 2.5

Esto significa que, en promedio, cada estado puede producir aproximadamente entre 2 y 3 movimientos legales.

El valor real puede cambiar dependiendo de la cantidad de paredes, puertas y caminos disponibles.

Para tener una idea del crecimiento, se puede calcular aproximadamente el número de nodos de un árbol usando:

    1 + b + b² + ... + b^d

donde d representa la profundidad de la solución.

Este crecimiento explica por qué las búsquedas pueden volverse costosas cuando aumenta la profundidad.

In [ ]:
b = 2.5

profundidades = [4, 8, 12]

for profundidad in profundidades:

    nodos = 0

    for nivel in range(profundidad + 1):

        nodos += b ** nivel

    print(
        "Profundidad:",
        profundidad,
        "Nodos aproximados:",
        round(nodos)
    )

# 11. Comparación con la estimación inicial

En la propuesta se estimó un factor de ramificación de aproximadamente 2.5.

Esta estimación solamente sirve como referencia, porque el laberinto no es un árbol uniforme.

Las paredes reducen los movimientos disponibles y las puertas pueden reducirlos todavía más cuando no se posee la llave.

Además, recoger una llave puede cambiar las posibilidades futuras del explorador.

Por esta razón, el número real de estados explorados depende de la forma específica de cada laberinto.

# 12. Límite de búsqueda

Para evitar que una instancia demasiado grande tarde demasiado tiempo, se utiliza como referencia un límite de aproximadamente:

    1,000,000 de estados.

Si una instancia supera este número, se considera demasiado grande para las pruebas iniciales.

Por esta razón se utilizan laberintos de dificultad progresiva en lugar de comenzar directamente con problemas demasiado grandes.

In [ ]:
limite = 1000000

for resultado in resultados_ultra:

    estados = resultado["Estados explorados"]

    if estados >= limite:

        print(
            resultado["Algoritmo"],
            "superó o alcanzó el límite."
        )

    else:

        print(
            resultado["Algoritmo"],
            "terminó dentro del límite."
        )

# 13. Tabla final de resultados

La siguiente tabla permite comparar los algoritmos en las diferentes instancias.

Las columnas principales son:

- Algoritmo
- Movimientos de la solución
- Estados explorados

La cantidad de movimientos permite observar la solución encontrada y la cantidad de estados explorados permite observar el trabajo realizado por cada algoritmo.

In [ ]:
def agregar_instancia(resultados, nombre):

    datos = []

    for resultado in resultados:

        datos.append({
            "Instancia": nombre,
            "Algoritmo": resultado["Algoritmo"],
            "Movimientos": resultado["Movimientos"],
            "Estados explorados": resultado["Estados explorados"]
        })

    return datos


todos_los_resultados = []

todos_los_resultados += agregar_instancia(
    resultados_facil,
    "Fácil"
)

todos_los_resultados += agregar_instancia(
    resultados_dificil,
    "Difícil"
)

todos_los_resultados += agregar_instancia(
    resultados_ultra,
    "Ultra difícil"
)

tabla_final = pd.DataFrame(
    todos_los_resultados
)

tabla_final

# 14. Análisis de los resultados

A partir de las pruebas realizadas se puede observar que los algoritmos no exploran necesariamente la misma cantidad de estados.

BFS explora los estados por niveles, por lo que puede revisar una cantidad considerable de estados antes de encontrar la salida.

DFS sigue una rama hasta llegar a un punto donde debe regresar. Por esta razón, dependiendo de la distribución del laberinto, puede encontrar una solución rápidamente o explorar una cantidad mayor de estados.

A* utiliza información adicional para decidir qué estados revisar primero. En el caso de A* con Manhattan, se utiliza la distancia hasta la salida como una guía.

La comparación entre A* con h=0 y A* con Manhattan permite observar el efecto de utilizar una heurística específica del problema.

# 15. Análisis de la heurística Manhattan

La distancia Manhattan solamente considera la posición actual y la posición de salida.

No considera las paredes ni las puertas.

Por esta razón, la distancia calculada puede ser menor que la distancia real que debe recorrer el explorador.

Por ejemplo, si la salida está a cinco casillas horizontalmente, la heurística puede indicar cinco movimientos, aunque exista una pared que obligue a realizar un recorrido más largo.

Esto hace que la heurística sea útil como estimación sin agregar demasiada complejidad al programa.

Además, permite comparar de forma sencilla A* con h=0 contra A* utilizando información del problema.

# 16. Dificultad de las instancias

Las instancias fueron diseñadas con dificultad creciente.

La instancia fácil contiene pocos obstáculos y permite verificar que la implementación funcione correctamente.

La instancia difícil agrega paredes, una llave y una puerta, haciendo que el explorador tenga que tomar decisiones.

La instancia ultra difícil utiliza un laberinto de 15 x 15 y varias llaves y puertas.

El objetivo de utilizar diferentes instancias es observar cómo cambia el comportamiento de los algoritmos cuando aumenta el espacio de búsqueda.

# 17. Conclusiones

En este proyecto se implementó un problema de búsqueda basado en un explorador que debe encontrar la salida de un laberinto utilizando llaves para atravesar determinadas puertas.

La representación del estado utilizada fue:

    (posición, llaves_recogidas)

Esta representación permite conservar únicamente la información necesaria para continuar la búsqueda.

Se implementaron y compararon BFS, DFS, A* con h=0 y A* con distancia Manhattan.

Las pruebas muestran que la forma en que se realiza la búsqueda puede cambiar considerablemente la cantidad de estados explorados.

También se observó que aumentar el tamaño y la dificultad del laberinto puede aumentar rápidamente el espacio de búsqueda.

La distancia Manhattan fue utilizada como una heurística sencilla que proporciona información sobre qué tan cerca se encuentra el explorador de la salida.

Finalmente, el proyecto permitió comprobar de manera práctica cómo diferentes estrategias de búsqueda pueden resolver el mismo problema explorando diferentes cantidades de estados.

# 18. Trabajo futuro

Como trabajo futuro se podrían generar automáticamente más laberintos con diferentes cantidades de paredes, llaves y puertas.

También se podrían realizar muchas pruebas para obtener promedios en lugar de utilizar solamente una instancia de cada dificultad.

Otra posibilidad sería comparar los resultados utilizando diferentes posiciones de las llaves y puertas.

Esto permitiría estudiar con mayor detalle cómo cambia el comportamiento de los algoritmos dependiendo de la estructura del laberinto.